## Functions, exit codes, and `set -euo pipefail`

**Functions** group commands and are called like commands. Use **`local`** for variables so they don't leak into the rest of the script:

```bash
greet() {
    local name="$1"
    echo "Hello, $name"
}
greet "Alice"
```

Inside a function, `$1`, `$@`, `$#` are the *function's* arguments. A function returns the exit code of its **last command** (use `return N` explicitly if needed) — so `is_root() { [[ $EUID -eq 0 ]]; }` just works as a condition.

**Exit codes** are how everything signals success: `0` = success, non-zero = failure, available as **`$?`**. Set your script's own with `exit 0` / `exit 1`. In a *function* use `return`, not `exit` (which kills the whole script).

**`set -euo pipefail` — the production preamble.** By default bash is forgiving: a failing command doesn't stop the script, an unset variable expands to empty, a broken pipeline stage is swallowed. Production scripts want the opposite — **fail fast, fail loud**:

```bash
#!/bin/bash
set -euo pipefail
```

- **`-e`** — exit the moment any command fails (no blundering on after a failed `cp`).
- **`-u`** — error on an unset variable, catching typos like `$Path` for `$PATH`.
- **`-o pipefail`** — a pipeline fails if *any* stage fails, not just the last one.

To *allow* a command to fail, check it explicitly: `if ! risky-cmd; then …`. And when a script misbehaves, **`bash -x script.sh`** (or `set -x`) prints each command as it runs, with variables expanded — the fastest way to see what a script is actually doing. A `trap 'echo "FAIL at line $LINENO"' ERR` reports exactly where it died.
